In [1]:
!pip install catboost xgboost pyswarm --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.4/303.4 MB 1.1 MB/s eta 0:00:00


In [2]:
# === Imports ===
import os, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from pyswarm import pso

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
df_2017_2019 = pd.read_csv("/content/drive/MyDrive/Network Maintenace - With & without maintenace/journal-encoded-data/No-main-encoded-dataset-2017-2019/df_encoded_no-main_2017_2019.csv")

In [4]:
df_2017_2019.shape

(6444, 23)

In [5]:
df_2017_2019.head(10)

,AADT_mean_x,AADT_Single_Unit_mean_x,AADT_Combination_mean_x,Future_AADT_mean_x,IRI_mean_x,Thickness_Rigid_mean_x,Thickness_Flexible_mean_x,Base_Thickness_mean_x,F_System_mode,Surface_Type_mode,...,Faulting_mean_x,IRI_mean_y,RHU_AV_x,FRZ_IDX_x,TEMP_AVG_x,PRECIPITATION_x,Age_x,Urban_Type_rural,Urban_Type_small urban,Urban_Type_urban
0,40961,820.0,2295.0,40924.0,64.0,11.5,0.0,15.0,1,3,...,0.0,71.5,76.2,535.4,8.5,1068.6,5,0.0,0.0,1.0
1,26595,878.0,186.0,37230.0,172.8,0.0,3.0,6.0,3,2,...,0.0,198.8,67.1,3.7,19.6,933.1,16,0.0,0.0,1.0
2,61960,744.0,2851.0,62225.0,102.8,9.0,0.0,18.0,2,3,...,0.1,123.0,76.2,519.1,8.7,1083.9,6,0.0,0.0,1.0
3,52563,1052.0,2997.0,52581.0,65.5,11.0,0.0,16.0,1,3,...,0.0,75.0,76.4,504.3,8.3,963.7,4,0.0,0.0,1.0
4,13206,211.0,1267.0,13222.0,55.0,0.0,5.0,24.0,1,2,...,0.0,68.0,78.5,769.2,6.5,1021.7,7,1.0,0.0,0.0
5,16978,238.0,1443.0,16999.0,53.3,0.0,5.0,24.0,1,2,...,0.0,55.0,78.5,769.2,6.5,1021.7,7,1.0,0.0,0.0
6,21647,455.0,5196.0,21703.0,58.0,12.0,0.0,18.0,1,3,...,0.0,61.0,78.6,272.6,9.4,1097.5,8,0.0,1.0,0.0
7,100449,2008.0,5725.0,100481.0,67.0,10.0,0.0,26.0,2,3,...,0.0,70.5,76.3,457.0,9.1,1050.3,7,0.0,0.0,1.0
8,11524,300.0,426.0,12624.0,239.0,0.0,6.0,6.0,5,2,...,0.0,250.0,78.8,295.2,9.0,1060.7,1,0.0,0.0,1.0
9,2582,46.0,42.0,2583.0,178.3,0.0,3.0,6.0,5,2,...,0.0,184.0,78.8,295.2,9.0,1060.7,9,0.0,0.0,1.0


In [6]:
df_2017_2019.isnull().sum()

,0
AADT_mean_x,0
AADT_Single_Unit_mean_x,0
AADT_Combination_mean_x,0
Future_AADT_mean_x,0
IRI_mean_x,0
Thickness_Rigid_mean_x,0
Thickness_Flexible_mean_x,0
Base_Thickness_mean_x,0
F_System_mode,0
Surface_Type_mode,0


In [7]:
df = df_2017_2019.dropna()

In [8]:
df.shape

(6432, 23)

In [9]:
df.columns

Index(['AADT_mean_x', 'AADT_Single_Unit_mean_x', 'AADT_Combination_mean_x',
       'Future_AADT_mean_x', 'IRI_mean_x', 'Thickness_Rigid_mean_x',
       'Thickness_Flexible_mean_x', 'Base_Thickness_mean_x', 'F_System_mode',
       'Surface_Type_mode', 'Base_Type_mode_x', 'Rutting_mean_x',
       'Cracking_Percent_mean_x', 'Faulting_mean_x', 'IRI_mean_y', 'RHU_AV_x',
       'FRZ_IDX_x', 'TEMP_AVG_x', 'PRECIPITATION_x', 'Age_x',
       'Urban_Type_rural', 'Urban_Type_small urban', 'Urban_Type_urban'],
      dtype='object')

In [10]:
df.isnull().sum()

,0
AADT_mean_x,0
AADT_Single_Unit_mean_x,0
AADT_Combination_mean_x,0
Future_AADT_mean_x,0
IRI_mean_x,0
Thickness_Rigid_mean_x,0
Thickness_Flexible_mean_x,0
Base_Thickness_mean_x,0
F_System_mode,0
Surface_Type_mode,0


In [11]:
df.columns

Index(['AADT_mean_x', 'AADT_Single_Unit_mean_x', 'AADT_Combination_mean_x',
       'Future_AADT_mean_x', 'IRI_mean_x', 'Thickness_Rigid_mean_x',
       'Thickness_Flexible_mean_x', 'Base_Thickness_mean_x', 'F_System_mode',
       'Surface_Type_mode', 'Base_Type_mode_x', 'Rutting_mean_x',
       'Cracking_Percent_mean_x', 'Faulting_mean_x', 'IRI_mean_y', 'RHU_AV_x',
       'FRZ_IDX_x', 'TEMP_AVG_x', 'PRECIPITATION_x', 'Age_x',
       'Urban_Type_rural', 'Urban_Type_small urban', 'Urban_Type_urban'],
      dtype='object')

In [12]:
num_dupes = df.duplicated(keep=False).sum()
print(f"Duplicate rows in df (all columns considered): {num_dupes}")

Duplicate rows in df (all columns considered): 2


In [13]:
# --- Remove duplicate rows across all columns ---
initial_rows = len(df)

df.drop_duplicates(inplace=True)   # modifies df in place

print(f"Duplicates removed: {initial_rows - len(df)}")
print(f"New DataFrame shape: {df.shape}")


Duplicates removed: 1
New DataFrame shape: (6431, 23)


/tmp/ipykernel_4753/3110374706.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop_duplicates(inplace=True)   # modifies df in place


In [14]:
df.shape

(6431, 23)

In [15]:
X = df.drop(columns=['IRI_mean_y'])  # Drop target column to keep 21 input features
y = df[['IRI_mean_y']]               # Target stays (N, 1)

In [16]:
y.shape

(6431, 1)

In [17]:
print(X.shape)  # Should be (4060, 21)
print(y.shape)  # Should be (4060, 1)

(6431, 22)
(6431, 1)


In [18]:
type(y)

pandas.core.frame.DataFrame

In [19]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)
print("y_train:", np.asarray(y_train).shape)

X_train: (4501, 22)
X_val  : (965, 22)
X_test : (965, 22)
y_train: (4501, 1)


In [20]:
import numpy as np

from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR

from xgboost import XGBRegressor
from catboost import CatBoostRegressor


# =========================
# Evaluation Function
# =========================
def evaluate_model(model, name, X_split, y_split, split_label="Test"):
    y_pred = model.predict(X_split).reshape(-1, 1)
    y_true = np.asarray(y_split).reshape(-1, 1)

    n = len(y_true)

    mse  = np.mean((y_true - y_pred) ** 2)
    rmse = np.sqrt(mse)

    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - (ss_res / ss_tot)

    mae = np.mean(np.abs(y_true - y_pred))

    eps = 1e-10
    mape = (100 / n) * np.sum(np.abs((y_true - y_pred) / (y_true + eps)))

    vaf = (1 - np.var(y_true - y_pred) / np.var(y_true)) * 100
    rsr = rmse / np.sqrt(np.mean((y_true - np.mean(y_true)) ** 2))
    wmape = (np.sum(np.abs(y_true - y_pred)) / (np.sum(np.abs(y_true)) + eps)) * 100

    a20index = np.mean(np.abs(y_true - y_pred) <= 0.2 * np.abs(y_true))

    print(f"🔹 {name} ({split_label}):")
    print(f"   R²       = {r2:.4f}")
    print(f"   MSE      = {mse:.4f}")
    print(f"   RMSE     = {rmse:.4f}")
    print(f"   MAE      = {mae:.4f}")
    print(f"   MAPE     = {mape:.4f}%")
    print(f"   VAF      = {vaf:.4f}%")
    print(f"   RSR      = {rsr:.4f}")
    print(f"   WMAPE    = {wmape:.4f}%")
    print(f"   a20index = {a20index:.4f}")

    return {"split": split_label, "mse": mse, "rmse": rmse, "r2": r2, "mae": mae,
            "mape": mape, "vaf": vaf, "rsr": rsr, "wmape": wmape, "a20index": a20index}


# Helper: always make y 1D for sklearn
y_train_1d = np.ravel(y_train)
y_val_1d   = np.ravel(y_val)   if 'y_val' in globals()   else None
y_test_1d  = np.ravel(y_test)


In [21]:

# =========================
# 1) Random Forest
# =========================
rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    {
        "n_estimators": [100, 200],
        "max_features": ["sqrt", "log2"],
        "max_depth": [None, 10, 20],
        "min_samples_split": [2, 10],
        "min_samples_leaf": [1, 4],
    },
    cv=5, scoring="r2", n_jobs=-1
)
rf_grid.fit(X_train, y_train_1d)
rf_best = rf_grid.best_estimator_

rf_train_metrics = evaluate_model(rf_best, "Random Forest", X_train, y_train_1d, "Train")
rf_test_metrics  = evaluate_model(rf_best, "Random Forest", X_test,  y_test_1d,  "Test")
rf_test_metrics

🔹 Random Forest (Train):
   R²       = 0.9819
   MSE      = 64.7541
   RMSE     = 8.0470
   MAE      = 4.8977
   MAPE     = 4.4630%
   VAF      = 98.1879%
   RSR      = 0.1346
   WMAPE    = 4.3617%
   a20index = 0.9891
🔹 Random Forest (Test):
   R²       = 0.8829
   MSE      = 388.0797
   RMSE     = 19.6997
   MAE      = 12.4706
   MAPE     = 11.8533%
   VAF      = 88.3205%
   RSR      = 0.3422
   WMAPE    = 11.2976%
   a20index = 0.8228


{'split': 'Test',
 'mse': np.float64(388.07974960648386),
 'rmse': np.float64(19.699739836010117),
 'r2': np.float64(0.8829200981643405),
 'mae': np.float64(12.470584196891192),
 'mape': np.float64(11.853298117129999),
 'vaf': np.float64(88.32051842877698),
 'rsr': np.float64(0.34216940517185274),
 'wmape': np.float64(11.297621785784685),
 'a20index': np.float64(0.8227979274611399)}

In [22]:
# =========================
# 2) Gradient Boosting
# =========================
gbr_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    {
        "n_estimators": [100, 200, 300],
        "learning_rate": [0.01, 0.05, 0.1],
        "max_depth": [3, 6, 10],
    },
    cv=5, scoring="r2", n_jobs=-1
)
gbr_grid.fit(X_train, y_train_1d)
gbr_best = gbr_grid.best_estimator_

gbr_train_metrics = evaluate_model(gbr_best, "Gradient Boosting", X_train, y_train_1d, "Train")
gbr_test_metrics  = evaluate_model(gbr_best, "Gradient Boosting", X_test,  y_test_1d,  "Test")

gbr_test_metrics

🔹 Gradient Boosting (Train):
   R²       = 0.9242
   MSE      = 270.9308
   RMSE     = 16.4600
   MAE      = 9.6015
   MAPE     = 8.2071%
   VAF      = 92.4169%
   RSR      = 0.2754
   WMAPE    = 8.5508%
   a20index = 0.9471
🔹 Gradient Boosting (Test):
   R²       = 0.9146
   MSE      = 282.9412
   RMSE     = 16.8209
   MAE      = 10.0704
   MAPE     = 8.7868%
   VAF      = 91.4688%
   RSR      = 0.2922
   WMAPE    = 9.1232%
   a20index = 0.9347


{'split': 'Test',
 'mse': np.float64(282.94123696328495),
 'rmse': np.float64(16.820857200609158),
 'r2': np.float64(0.9146393691438106),
 'mae': np.float64(10.070434610047055),
 'mape': np.float64(8.786829901455935),
 'vaf': np.float64(91.46877830242596),
 'rsr': np.float64(0.2921654169407964),
 'wmape': np.float64(9.123226277655066),
 'a20index': np.float64(0.9347150259067357)}

In [23]:
# =========================
# 3) XGBoost
# =========================
xgb_grid = GridSearchCV(
    XGBRegressor(random_state=42, verbosity=0),
    {
        "n_estimators": [100, 200, 300],
        "max_depth": [3, 6, 10],
        "learning_rate": [0.01, 0.05, 0.1],
    },
    cv=5, scoring="r2", n_jobs=-1
)
xgb_grid.fit(X_train, y_train_1d)
xgb_best = xgb_grid.best_estimator_

xgb_train_metrics = evaluate_model(xgb_best, "XGBoost", X_train, y_train_1d, "Train")
xgb_test_metrics  = evaluate_model(xgb_best, "XGBoost", X_test,  y_test_1d,  "Test")

xgb_test_metrics

🔹 XGBoost (Train):
   R²       = 0.9282
   MSE      = 256.4560
   RMSE     = 16.0142
   MAE      = 9.3193
   MAPE     = 7.9485%
   VAF      = 92.8220%
   RSR      = 0.2679
   WMAPE    = 8.2995%
   a20index = 0.9485
🔹 XGBoost (Test):
   R²       = 0.9179
   MSE      = 272.0843
   RMSE     = 16.4950
   MAE      = 9.9352
   MAPE     = 8.6749%
   VAF      = 91.7973%
   RSR      = 0.2865
   WMAPE    = 9.0007%
   a20index = 0.9326


{'split': 'Test',
 'mse': np.float64(272.0842916151191),
 'rmse': np.float64(16.494977769464228),
 'r2': np.float64(0.9179148044039274),
 'mae': np.float64(9.935197690657384),
 'mape': np.float64(8.674863519930497),
 'vaf': np.float64(91.79734832809581),
 'rsr': np.float64(0.28650514061020366),
 'wmape': np.float64(9.000709518005582),
 'a20index': np.float64(0.9326424870466321)}

In [24]:
# =========================
# 4) CatBoost
# =========================
cat_grid = GridSearchCV(
    CatBoostRegressor(verbose=0, random_state=42),
    {
        "iterations": [300, 800],
        "depth": [6, 8, 10],
        "learning_rate": [0.01, 0.05, 0.1],
    },
    cv=5, scoring="r2", n_jobs=-1
)
cat_grid.fit(X_train, y_train_1d)
cat_best = cat_grid.best_estimator_

cat_train_metrics = evaluate_model(cat_best, "CatBoost", X_train, y_train_1d, "Train")
cat_test_metrics  = evaluate_model(cat_best, "CatBoost", X_test,  y_test_1d,  "Test")

cat_test_metrics


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


🔹 CatBoost (Train):
   R²       = 0.9544
   MSE      = 163.0547
   RMSE     = 12.7693
   MAE      = 7.9202
   MAPE     = 6.9285%
   VAF      = 95.4363%
   RSR      = 0.2136
   WMAPE    = 7.0534%
   a20index = 0.9625
🔹 CatBoost (Test):
   R²       = 0.9182
   MSE      = 271.0719
   RMSE     = 16.4643
   MAE      = 9.9992
   MAPE     = 8.8077%
   VAF      = 91.8258%
   RSR      = 0.2860
   WMAPE    = 9.0587%
   a20index = 0.9192


{'split': 'Test',
 'mse': np.float64(271.0719128940925),
 'rmse': np.float64(16.464261686880846),
 'r2': np.float64(0.9182202292589952),
 'mae': np.float64(9.999231296239106),
 'mape': np.float64(8.80771513340209),
 'vaf': np.float64(91.82584560718793),
 'rsr': np.float64(0.2859716257620759),
 'wmape': np.float64(9.058720229133515),
 'a20index': np.float64(0.9191709844559586)}

In [25]:
# =========================
# 5) Linear Regression (optional scaling)
# =========================
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LinearRegression())
])

# LinearRegression has no real hyperparams; use it directly (or grid with empty dict)
lr_pipe.fit(X_train, y_train_1d)

lr_train_metrics = evaluate_model(lr_pipe, "Linear Regression", X_train, y_train_1d, "Train")
lr_test_metrics  = evaluate_model(lr_pipe, "Linear Regression", X_test,  y_test_1d,  "Test")

lr_test_metrics


🔹 Linear Regression (Train):
   R²       = 0.8999
   MSE      = 357.5773
   RMSE     = 18.9097
   MAE      = 10.9314
   MAPE     = 9.2888%
   VAF      = 89.9917%
   RSR      = 0.3164
   WMAPE    = 9.7351%
   a20index = 0.9340
🔹 Linear Regression (Test):
   R²       = 0.9083
   MSE      = 304.0482
   RMSE     = 17.4370
   MAE      = 10.4315
   MAPE     = 9.1739%
   VAF      = 90.8348%
   RSR      = 0.3029
   WMAPE    = 9.4503%
   a20index = 0.9368


{'split': 'Test',
 'mse': np.float64(304.04823948866976),
 'rmse': np.float64(17.436979081500034),
 'r2': np.float64(0.908271590906933),
 'mae': np.float64(10.431505020483188),
 'mape': np.float64(9.173934369233397),
 'vaf': np.float64(90.8348019974444),
 'rsr': np.float64(0.30286698250728317),
 'wmape': np.float64(9.45033500574195),
 'a20index': np.float64(0.9367875647668393)}

In [26]:
# =========================
# 6) SVR (scaling REQUIRED)
# =========================
svr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR())
])

svr_grid = GridSearchCV(
    svr_pipe,
    {
        "svr__kernel": ["rbf", "linear"],
        "svr__C": [1, 10, 100],
        "svr__epsilon": [0.01, 0.1, 0.5],
        "svr__gamma": ["scale", "auto"],
    },
    cv=5, scoring="r2", n_jobs=-1
)
svr_grid.fit(X_train, y_train_1d)
svr_best = svr_grid.best_estimator_

svr_train_metrics = evaluate_model(svr_best, "SVR", X_train, y_train_1d, "Train")
svr_test_metrics  = evaluate_model(svr_best, "SVR", X_test,  y_test_1d,  "Test")

svr_test_metrics


🔹 SVR (Train):
   R²       = 0.9211
   MSE      = 281.7903
   RMSE     = 16.7866
   MAE      = 7.1628
   MAPE     = 5.2947%
   VAF      = 92.4517%
   RSR      = 0.2808
   WMAPE    = 6.3790%
   a20index = 0.9511
🔹 SVR (Test):
   R²       = 0.9023
   MSE      = 323.9757
   RMSE     = 17.9993
   MAE      = 9.9822
   MAPE     = 8.4440%
   VAF      = 90.4936%
   RSR      = 0.3126
   WMAPE    = 9.0433%
   a20index = 0.9088


{'split': 'Test',
 'mse': np.float64(323.97567177382047),
 'rmse': np.float64(17.999324203253313),
 'r2': np.float64(0.9022596775871888),
 'mae': np.float64(9.98219086679098),
 'mape': np.float64(8.443955174390736),
 'vaf': np.float64(90.49357763601508),
 'rsr': np.float64(0.3126344869217264),
 'wmape': np.float64(9.043282594141221),
 'a20index': np.float64(0.9088082901554404)}

In [27]:
# =========================
# Collect all metrics (easy to save to df/excel later)
# =========================
all_metrics = {
    "RandomForest": {"train": rf_train_metrics, "test": rf_test_metrics},
    "GradientBoosting": {"train": gbr_train_metrics, "test": gbr_test_metrics},
    "XGBoost": {"train": xgb_train_metrics, "test": xgb_test_metrics},
    "CatBoost": {"train": cat_train_metrics, "test": cat_test_metrics},
    "LinearRegression": {"train": lr_train_metrics, "test": lr_test_metrics},
    "SVR": {"train": svr_train_metrics, "test": svr_test_metrics},
}

all_metrics


{'RandomForest': {'train': {'split': 'Train',
   'mse': np.float64(64.75410876387211),
   'rmse': np.float64(8.046993771830081),
   'r2': np.float64(0.9818759321281199),
   'mae': np.float64(4.8976779895472955),
   'mape': np.float64(4.463006231754926),
   'vaf': np.float64(98.18790560810751),
   'rsr': np.float64(0.13462565829692416),
   'wmape': np.float64(4.361700488604265),
   'a20index': np.float64(0.9891135303265941)},
  'test': {'split': 'Test',
   'mse': np.float64(388.07974960648386),
   'rmse': np.float64(19.699739836010117),
   'r2': np.float64(0.8829200981643405),
   'mae': np.float64(12.470584196891192),
   'mape': np.float64(11.853298117129999),
   'vaf': np.float64(88.32051842877698),
   'rsr': np.float64(0.34216940517185274),
   'wmape': np.float64(11.297621785784685),
   'a20index': np.float64(0.8227979274611399)}},
 'GradientBoosting': {'train': {'split': 'Train',
   'mse': np.float64(270.93077347225307),
   'rmse': np.float64(16.459974892819645),
   'r2': np.float64(